In [0]:
%run ../configs/config

In [0]:
clients = f'{bronze_schema}.clients'
clients_df = spark.table(clients)

In [0]:
clients_silver_df = (clients_df
    .withColumn('birth_month_raw', 
        (F.col('birth_number') / 100).cast('int') % 100
    )
    .withColumn('gender',
        F.when(F.col('birth_month_raw') > 50, 'Female')
        .otherwise('Male')
    )
    .withColumn('birth_month',
        F.when(F.col('birth_month_raw') > 50, F.col('birth_month_raw') - 50)
        .otherwise(F.col('birth_month_raw'))
    )
    .withColumn('birth_year',
        (F.col('birth_number') / 10000).cast('int') + 1900
    )
    .withColumn('birth_day',
        F.col('birth_number') % 100
    )
    .drop('birth_month_raw')
)

In [0]:
display(clients_silver_df)

In [0]:
clients_final_df = add_metadata_columns(clients_silver_df)

In [0]:
(
    clients_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(f'{silver_schema}.clients')
)